In [1]:
# Installing Dependencies
!pip install -qU langchain langchain-community langchain-text-splitters langchain-google-genai pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires request

In [3]:
# Imports
import os
from google.colab import userdata, files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Securely fetch the API key from Colab Secrets
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
    print("✅ API Key loaded successfully.")
except Exception as e:
    print("❌ Error: Please set GOOGLE_API_KEY in the Colab Secrets tab.")

✅ API Key loaded successfully.


In [5]:
# Cell 3
import os

# Define the exact name of the file you uploaded to Colab
pdf_filename = "intro-to-ml.pdf"

# Colab's default working directory is /content/
pdf_path = f"/content/intro-to-ml.pdf"

# Verify the file exists before proceeding
if os.path.exists(pdf_path):
    print(f"✅ File successfully located at: {pdf_path}")
    print(f"File size: {os.path.getsize(pdf_path) / (1024*1024):.2f} MB")
else:
    print(f"❌ Error: Could not find '{pdf_filename}'.")
    print("Please ensure you dragged and dropped the file into the Files panel on the left.")

✅ File successfully located at: /content/intro-to-ml.pdf
File size: 6.73 MB


In [6]:
!pip install -qU langchain-huggingface sentence-transformers

In [7]:
!pip install -qU pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 88.6 MB/s eta 0:00:00


In [8]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings


loader = PyMuPDFLoader(pdf_path)
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(pages)
print(f" Created {len(chunks)} chunks.")



 Created 663 chunks.


In [9]:
# Using HuggingFace  Embedding Model
print("\n2. Initializing HuggingFace Embedding Model...")
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("\n3. Running Quarantine Embedding Loop...")
valid_texts = []
valid_metadatas = []
valid_embeddings = []

# Process chunks one by one. If one crashes, we catch the error and skip it!
for i, doc in enumerate(chunks):
    text = str(doc.page_content).replace('\x00', '').replace('\ufffd', '').strip()

    if len(text) < 15:
        continue

    try:
        # Attempt to embed just this single piece of text
        emb = embeddings_model.embed_query(text)

        # If it succeeds, save everything
        valid_texts.append(text)
        valid_embeddings.append(emb)
        valid_metadatas.append(doc.metadata)

    except Exception as e:
        # IF IT CRASHES, we catch it here instead of breaking the notebook
        print(f"⚠️ Caught and skipped poisoned chunk at index {i}!")

print(f"\n✅ Successfully embedded {len(valid_texts)} safe chunks!")

print("\n4. Building and saving FAISS index...")
# We use from_embeddings because we already generated the math vectors safely
text_embeddings = list(zip(valid_texts, valid_embeddings))
vectorstore = FAISS.from_embeddings(text_embeddings, embeddings_model, metadatas=valid_metadatas)

vectorstore.save_local("faiss_huggingface_index")
print("💾 FAISS index saved securely as 'faiss_huggingface_index'!")


2. Initializing HuggingFace Embedding Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


3. Running Quarantine Embedding Loop...

✅ Successfully embedded 663 safe chunks!

4. Building and saving FAISS index...
💾 FAISS index saved securely as 'faiss_huggingface_index'!


In [10]:
# 5. Setup Retriever and Gemini LLM

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)

system_prompt = (
    "You are an expert Machine Learning assistant."
    "Use the following pieces of retrieved context to answer the question."
    "If you don't know the answer or if the answer is not contained in the context, "
    "just say that you don't know. Do NOT make up an answer or use outside knowledge."
    "Keep the answer concise and strictly factual.\n\n"
    "Context: {context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [12]:
# END-TO-END RAG Q&A APPLICATION

while True:
    # 1. Accept user query
    query = input("\nAsk a question about Machine Learning: ")

    if query.lower() in ['exit', 'quit']:
        print("Goodbye!")
        break

    if not query.strip():
        continue

    print("\n🔍 Searching textbook and thinking...")

    # Step 1: Retrieve the top-k most relevant chunks from FAISS
    retrieved_docs = retriever.invoke(query)

    # Step 2: Stitch the retrieved text together into one big "Context" string
    context_text = "\n\n---\n\n".join([doc.page_content for doc in retrieved_docs])

    # Step 3: Inject the context and the user's query into our strict prompt template
    formatted_prompt = prompt_template.format_messages(
        context=context_text,
        input=query
    )

    # Step 4: Send the prompt to the LLM
    llm_response = llm.invoke(formatted_prompt)

    # Display the generated answer
    print("\n💡 Answer:")
    print(llm_response.content)

    #Evidence of grounding (Show the sources)
    print("📚 Evidence / Retrieved Context")
    print("-"*40)
    for i, doc in enumerate(retrieved_docs):
        # Extract page number
        page_num = doc.metadata.get("page", "Unknown")

        snippet = doc.page_content[:200].replace('\n', ' ')

        print(f"🔹 Source {i+1} (Page {page_num}): {snippet}...\n")


Ask a question about Machine Learning: what is ml

🔍 Searching textbook and thinking...

💡 Answer:
I don't know.
📚 Evidence / Retrieved Context
----------------------------------------
🔹 Source 1 (Page 130): Figure 2-54. Heat map of the first layer weights in a neural network learned on the Breast Cancer dataset One possible inference we can make is that features that have very small weights for all of th...

🔹 Source 2 (Page 131): which works well in most situations but is quite sensitive to the scaling of the data (so it is important to always scale your data to 0 mean and unit variance). The other one is 'l-bfgs', which is qu...

🔹 Source 3 (Page 121): In[92]: mglearn.plots.plot_two_hidden_layer_graph() Figure 2-47. A multilayer perceptron with two hidden layers Having large neural networks made up of many of these layers of computation is what insp...

🔹 Source 4 (Page 9): how to use the large array of models already implemented in scikit-learn and other libraries. Why We Wrote Th

## Project Summary: Retrieval-Augmented Generation (RAG) for Machine Learning Knowledge

This project developed an end-to-end Retrieval-Augmented Generation (RAG) system to answer technical questions about Machine Learning and Deep Learning using a PDF version of a machine learning book.

The system was built in several key parts:

**1. Data Understanding & Preprocessing:**
- The project began by loading a provided PDF document (`intro-to-ml.pdf`).
- Text was extracted from all pages of the PDF.
- The extracted text was then split into manageable `chunks` using `RecursiveCharacterTextSplitter`, with a `chunk_size` of 1500 and `chunk_overlap` of 150, to optimize for relevant content retrieval.

**2. Embedding & Vector Database:**
- `HuggingFaceEmbeddings` (specifically `all-MiniLM-L6-v2`) were used to generate numerical embeddings for each text chunk.
- These embeddings, along with their associated metadata (like page numbers), were stored in a `FAISS` vector database, which was saved locally as `'faiss_huggingface_index'`.

**3. Retrieval Pipeline:**
- A retriever was set up from the FAISS vector store to fetch the top 5 (`k=5`) most relevant text chunks based on a user's query.

**4. Answer Generation (RAG):**
- A `ChatGoogleGenerativeAI` model (`gemini-2.5-flash`) was integrated as the Large Language Model (LLM).
- A `ChatPromptTemplate` was designed to ensure that the LLM's responses were strictly grounded in the retrieved context, preventing hallucination and ensuring factual accuracy.

**5. End-to-End RAG Application:**
- A functional Q&A loop was implemented, allowing users to input questions about Machine Learning.
- For each query, the system first retrieved relevant information from the FAISS database, then used this context to generate a concise and factual answer via the Gemini LLM.
- The application also provided evidence by displaying snippets of the retrieved context, including page numbers, to demonstrate grounding.

This system effectively combines retrieval capabilities with a generative AI model to provide accurate and context-aware answers from a specialized knowledge base.